In [0]:
import pandas as pd
import re
from pyspark.sql import SparkSession

# ---------------------------
# 0) Paths
# ---------------------------
CSV_LOCAL = "../../data/raw/Toronto_weather_data.csv"

BRONZE_PATH = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/weather"
SILVER_PATH = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly"

# ---------------------------
# 1) Read CSV with pandas
# ---------------------------
df_raw = pd.read_csv(CSV_LOCAL, sep=";", encoding="utf-8-sig")

print("Rows:", len(df_raw))
print(df_raw.head())

# Fix encoding artifact
df_raw.columns = [c.replace("Â", "").strip() for c in df_raw.columns]

# ---------------------------
# 2) Rename temperature cols safely
# ---------------------------
for c in df_raw.columns:
    if "temperature_2m" in c and "apparent" not in c:
        df_raw = df_raw.rename(columns={c: "temperature_2m_celsius"})
    if "apparent_temperature" in c:
        df_raw = df_raw.rename(columns={c: "apparent_temperature_celsius"})

# ---------------------------
# 3) Normalize hour
# ---------------------------
def normalize_hour(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    m = re.match(r"^(\d{1,2})", s)
    if not m:
        return None
    h = int(m.group(1))
    if 0 <= h <= 23:
        return f"{h:02d}:00"
    return None

df_raw["hour"] = df_raw["hour"].apply(normalize_hour)

# ---------------------------
# 4) Convert to Spark DF
# ---------------------------
spark_df = spark.createDataFrame(df_raw)

# ---------------------------
# 5) Write Bronze
# ---------------------------
spark_df.write.mode("overwrite").parquet(BRONZE_PATH)

print("✅ Bronze written:", BRONZE_PATH)

# ---------------------------
# 6) Basic Silver cleaning
# ---------------------------
from pyspark.sql import functions as F

silver_df = (
    spark_df
    .filter(
        F.col("year").isNotNull() &
        F.col("month").between(1,12) &
        F.col("day").between(1,31) &
        F.col("hour").isNotNull()
    )
)

silver_df.write.mode("overwrite").parquet(SILVER_PATH)

print("✅ Silver written:", SILVER_PATH)

# ---------------------------
# 7) Validation
# ---------------------------
print("Bronze rows:", spark.read.parquet(BRONZE_PATH).count())
print("Silver rows:", spark.read.parquet(SILVER_PATH).count())

display(silver_df.limit(10))


Rows: 17544
   year  month  day  hour  temperature_2m (°C)  apparent_temperature (°C)
0  2022     10    1  0:00                 12.3                       10.6
1  2022     10    1  1:00                 12.6                       11.3
2  2022     10    1  2:00                 11.6                       10.2
3  2022     10    1  3:00                 10.8                        9.1
4  2022     10    1  4:00                  9.9                        8.2
✅ Bronze written: dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/weather
✅ Silver written: dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly
Bronze rows: 17544
Silver rows: 17544


year,month,day,hour,temperature_2m_celsius,apparent_temperature_celsius
2022,10,1,00:00,12.3,10.6
2022,10,1,01:00,12.6,11.3
2022,10,1,02:00,11.6,10.2
2022,10,1,03:00,10.8,9.1
2022,10,1,04:00,9.9,8.2
2022,10,1,05:00,9.7,7.6
2022,10,1,06:00,9.6,7.7
2022,10,1,07:00,9.5,7.5
2022,10,1,08:00,10.1,7.8
2022,10,1,09:00,10.4,7.9


In [0]:
weather = spark.read.parquet("dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly")

display(
  weather.select("hour").groupBy("hour").count().orderBy("hour").limit(30)
)


hour,count
00:00,731
01:00,731
02:00,731
03:00,731
04:00,731
05:00,731
06:00,731
07:00,731
08:00,731
09:00,731


In [0]:
from pyspark.sql import functions as F

display(
  weather.select(
    F.count(F.when(F.col("year").isNull(), True)).alias("null_year"),
    F.count(F.when(F.col("month").isNull(), True)).alias("null_month"),
    F.count(F.when(F.col("day").isNull(), True)).alias("null_day"),
    F.count(F.when(F.col("hour").isNull(), True)).alias("null_hour")
  )
)


null_year,null_month,null_day,null_hour
0,0,0,0


In [0]:
display(
  weather.groupBy("year","month").count().orderBy("year","month")
)


year,month,count
2022,10,744
2022,11,720
2022,12,744
2023,1,744
2023,2,672
2023,3,744
2023,4,720
2023,5,744
2023,6,720
2023,7,744
